# Mapping the UK Circular Economy Research Ecosystem: Inputs and Outputs

**Clean GitHub version of the dissertation analysis notebook**

This notebook reproduces the final disciplinary classification and the principal analyses used in the dissertation:

1. project disciplinary classification;
2. RQ1 funding analysis;
3. H1 funding regression;
4. RQ2 project–output disciplinary alignment;
5. H2 permutation test;
6. interdisciplinarity analysis and maturity robustness checks;
7. final figures and analytical tables.

Exploratory checks, duplicated figure versions, temporary diagnostics, local Windows paths and intermediate debugging cells from the original notebook have been removed.

> **Important:** Scopus was used only as a supplementary bibliographic source. Raw Scopus exports should not be redistributed publicly unless permitted by the applicable licence.

## 1. Repository structure and required files

This notebook uses paths relative to the GitHub repository.

Recommended structure:

```text
uk-circular-economy-research-ecosystem/
├── README.md
├── requirements.txt
├── notebooks/
│   └── CE_research_ecosystem_analysis.ipynb
├── data/
│   ├── project_level_analysis.csv
│   ├── gtr_project_labels_8cat_10pct.csv
│   ├── gtr_project_discipline_summary_8cat_final.csv
│   ├── openalex_field_crosswalk_8cat_final.csv
│   ├── manual_coding_sample_200_completed.xlsx
│   └── integrated_outputs_final.csv
├── results/
└── figures/
```

The notebook will also create the final project classification file and the RQ1/RQ2 result tables.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
)

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import mcnemar
from scipy.stats import wilcoxon

RANDOM_STATE = 42

# When the notebook is run from notebooks/, the repository root is one level up.
REPO_DIR = Path.cwd()
if REPO_DIR.name == "notebooks":
    REPO_DIR = REPO_DIR.parent

DATA_DIR = REPO_DIR / "data"
RESULTS_DIR = REPO_DIR / "results"
FIGURES_DIR = REPO_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

discipline_classes = [
    "Engineering & Manufacturing",
    "Materials & Chemistry",
    "Environmental & Earth Sciences",
    "Life, Agricultural & Health Sciences",
    "Social Sciences, Business & Economics",
    "Energy",
    "Computer & Quantitative Sciences",
    "Arts, Humanities & Behavioural Sciences",
]

print("Repository:", REPO_DIR)
print("Data:", DATA_DIR)

## 2. Load processed source data

The public repository starts from processed research-project and publication-link files rather than repeating API harvesting. This keeps the supplementary material focused on the analysis reported in the dissertation.

In [ ]:
required_files = {
    "project_level": DATA_DIR / "project_level_analysis.csv",
    "official_labels": DATA_DIR / "gtr_project_labels_8cat_10pct.csv",
    "official_summary": DATA_DIR / "gtr_project_discipline_summary_8cat_final.csv",
    "field_crosswalk": DATA_DIR / "openalex_field_crosswalk_8cat_final.csv",
    "human_gold": DATA_DIR / "manual_coding_sample_200_completed.xlsx",
    "integrated_outputs": DATA_DIR / "integrated_outputs_final.csv",
}

missing = [str(path) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "The following required files are missing:\n- " + "\n- ".join(missing)
    )

project_level = pd.read_csv(required_files["project_level"], low_memory=False)
final_label_long = pd.read_csv(required_files["official_labels"])
gtr_project_discipline_summary_8 = pd.read_csv(required_files["official_summary"])
openalex_field_table = pd.read_csv(required_files["field_crosswalk"])
gold = pd.read_excel(required_files["human_gold"])
final_integrated = pd.read_csv(required_files["integrated_outputs"], low_memory=False)

for df in [
    project_level,
    final_label_long,
    gtr_project_discipline_summary_8,
    gold,
    final_integrated,
]:
    if "project_id" in df.columns:
        df["project_id"] = df["project_id"].astype(str)

print("Projects:", len(project_level))
print("Official-labelled projects:", gtr_project_discipline_summary_8["project_id"].nunique())
print("Human-coded projects:", len(gold))
print("Project-publication links:", len(final_integrated))

## 3. Disciplinary classification

The dissertation uses eight common disciplinary categories. Projects with official GtR subject information retain mapped official labels. A human-coded gold standard of 200 projects is used to evaluate the text classifier.

For clarity, this GitHub notebook uses the **locked final settings** selected during model development:

- TF-IDF features;
- one-vs-rest logistic regression;
- `C = 2.0`;
- balanced class weights;
- discipline-specific probability thresholds.

The exploratory tuning cells from the working notebook are not repeated here.

In [ ]:
# Build official 443-project training dataset
official_ids = set(gtr_project_discipline_summary_8["project_id"].astype(str))

training_projects = project_level[
    project_level["project_id"].astype(str).isin(official_ids)
].copy()

training_projects["title_clean"] = training_projects["title"].fillna("").astype(str).str.strip()
training_projects["abstract_clean"] = (
    training_projects["abstract_text"].fillna("").astype(str).str.strip()
)

# Preserve the final training-text construction used in the working analysis.
training_projects["classification_text"] = (
    training_projects["title_clean"]
    + " "
    + training_projects["title_clean"]
    + " "
    + training_projects["abstract_clean"]
).str.strip()

label_matrix = (
    final_label_long.assign(value=1)
    .pivot_table(
        index="project_id",
        columns="common_discipline",
        values="value",
        aggfunc="max",
        fill_value=0,
    )
    .reindex(columns=discipline_classes, fill_value=0)
)
label_matrix.index = label_matrix.index.astype(str)

training_data = (
    training_projects[["project_id", "title", "classification_text"]]
    .drop_duplicates("project_id")
    .set_index("project_id")
    .join(label_matrix, how="inner")
    .reset_index()
)

print("Official training projects:", len(training_data))
print("Zero-label official projects:", int((training_data[discipline_classes].sum(axis=1) == 0).sum()))

In [ ]:
# Quality-check the 200 human-coded projects
for c in discipline_classes:
    gold[c] = pd.to_numeric(gold[c], errors="coerce")

if gold[discipline_classes].isna().any().any():
    raise ValueError("Human-coded gold standard contains missing disciplinary labels.")

if not set(np.unique(gold[discipline_classes].to_numpy())) <= {0, 1}:
    raise ValueError("Human-coded labels must be binary 0/1 values.")

if (gold[discipline_classes].sum(axis=1) == 0).any():
    raise ValueError("At least one human-coded project has no discipline label.")

# Reproduce the 140/60 development/external-test split.
if "sampling_group" in gold.columns:
    funder_dummies = pd.get_dummies(gold["sampling_group"], prefix="funder").astype(int)
    Y_strat = np.hstack([
        gold[discipline_classes].astype(int).to_numpy(),
        funder_dummies.to_numpy(),
    ])
else:
    Y_strat = gold[discipline_classes].astype(int).to_numpy()

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42,
)

dev_idx, test_idx = next(splitter.split(np.zeros(len(gold)), Y_strat))
gold_dev = gold.iloc[dev_idx].copy().reset_index(drop=True)
gold_test = gold.iloc[test_idx].copy().reset_index(drop=True)

print("Human-coded development set:", len(gold_dev))
print("External validation set:", len(gold_test))

In [ ]:
# Build the 583-project model-development dataset
text_columns = [
    "title",
    "abstract_text",
    "tech_abstract_text",
    "potential_impact",
]

human_dev = gold_dev.copy()
available_human_text = [c for c in text_columns if c in human_dev.columns]
human_dev["classification_text"] = (
    human_dev[available_human_text]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
human_dev["label_source"] = "human_coded"

official_dev = training_data.copy()
official_dev["label_source"] = "GtR_official"

base_columns = ["project_id", "classification_text", "label_source"] + discipline_classes

model_dev_583 = pd.concat(
    [
        official_dev[base_columns],
        human_dev[base_columns],
    ],
    ignore_index=True,
)

print("Model-development projects:", len(model_dev_583))
print(model_dev_583["label_source"].value_counts().to_string())

In [ ]:
# Locked final classifier settings from the dissertation analysis
FINAL_C = 2.0
FINAL_CLASS_WEIGHT = "balanced"

final_thresholds = np.array([
    0.425,  # Engineering & Manufacturing
    0.450,  # Materials & Chemistry
    0.425,  # Environmental & Earth Sciences
    0.425,  # Life, Agricultural & Health Sciences
    0.450,  # Social Sciences, Business & Economics
    0.375,  # Energy
    0.400,  # Computer & Quantitative Sciences
    0.325,  # Arts, Humanities & Behavioural Sciences
])

# External validation on the locked 60-project test set
train_text = model_dev_583["classification_text"].fillna("").astype(str).to_numpy()
Y_train = model_dev_583[discipline_classes].astype(int).to_numpy()

external_test = gold_test.copy()
available_test_text = [c for c in text_columns if c in external_test.columns]
external_test["classification_text"] = (
    external_test[available_test_text]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

test_text = external_test["classification_text"].to_numpy()
Y_test = external_test[discipline_classes].astype(int).to_numpy()

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=40000,
    sublinear_tf=True,
)

X_train = vectorizer.fit_transform(train_text)
X_test = vectorizer.transform(test_text)

model = OneVsRestClassifier(
    LogisticRegression(
        C=FINAL_C,
        class_weight=FINAL_CLASS_WEIGHT,
        max_iter=3000,
        solver="liblinear",
        random_state=42,
    )
)
model.fit(X_train, Y_train)

test_prob = model.predict_proba(X_test)
test_pred = (test_prob >= final_thresholds).astype(int)

# Fallback for any project receiving no label.
for i in np.where(test_pred.sum(axis=1) == 0)[0]:
    test_pred[i, np.argmax(test_prob[i])] = 1

validation_metrics = pd.DataFrame({
    "metric": ["Micro F1", "Macro F1", "Weighted F1", "Hamming loss", "Exact match"],
    "value": [
        f1_score(Y_test, test_pred, average="micro", zero_division=0),
        f1_score(Y_test, test_pred, average="macro", zero_division=0),
        f1_score(Y_test, test_pred, average="weighted", zero_division=0),
        hamming_loss(Y_test, test_pred),
        (test_pred == Y_test).all(axis=1).mean(),
    ],
})

validation_metrics

In [ ]:
# Train the final model on all 443 official + 200 human-coded projects
human_all = gold.copy()
available_text_columns = [c for c in text_columns if c in human_all.columns]
human_all["classification_text"] = (
    human_all[available_text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
human_all["label_source"] = "human_coded"

official_all = training_data.copy()
official_all["label_source"] = "GtR_official"

final_labelled = pd.concat(
    [
        official_all[base_columns],
        human_all[base_columns],
    ],
    ignore_index=True,
).drop_duplicates("project_id", keep="last").reset_index(drop=True)

labelled_ids = set(final_labelled["project_id"].astype(str))
remaining_projects = project_level[
    ~project_level["project_id"].astype(str).isin(labelled_ids)
].copy().reset_index(drop=True)

remaining_text_columns = [c for c in text_columns if c in remaining_projects.columns]
remaining_projects["classification_text"] = (
    remaining_projects[remaining_text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

final_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    max_features=40000,
    sublinear_tf=True,
)

X_final_train = final_vectorizer.fit_transform(final_labelled["classification_text"])
X_remaining = final_vectorizer.transform(remaining_projects["classification_text"])
Y_final_train = final_labelled[discipline_classes].astype(int).to_numpy()

final_model = OneVsRestClassifier(
    LogisticRegression(
        C=FINAL_C,
        class_weight=FINAL_CLASS_WEIGHT,
        max_iter=3000,
        solver="liblinear",
        random_state=42,
    )
)
final_model.fit(X_final_train, Y_final_train)

remaining_probs = final_model.predict_proba(X_remaining)
remaining_pred = (remaining_probs >= final_thresholds).astype(int)

for i in np.where(remaining_pred.sum(axis=1) == 0)[0]:
    remaining_pred[i, np.argmax(remaining_probs[i])] = 1

for j, discipline in enumerate(discipline_classes):
    remaining_projects[discipline] = remaining_pred[:, j]
    remaining_projects[f"prob_{discipline}"] = remaining_probs[:, j]

remaining_projects["label_source"] = "model_predicted"

print("Final labelled projects:", len(final_labelled))
print("Model-predicted projects:", len(remaining_projects))

In [ ]:
# Build the final 1,854-project dataset
labelled_final = final_labelled.copy()

for discipline in discipline_classes:
    prob_col = f"prob_{discipline}"
    if prob_col not in labelled_final.columns:
        labelled_final[prob_col] = np.nan

classification_columns = (
    ["project_id", "label_source"]
    + discipline_classes
    + [f"prob_{d}" for d in discipline_classes]
)

final_classification = pd.concat(
    [
        labelled_final[classification_columns],
        remaining_projects[classification_columns],
    ],
    ignore_index=True,
)

final_classification["n_common_disciplines"] = (
    final_classification[discipline_classes].astype(int).sum(axis=1)
)
final_classification["project_interdisciplinary"] = (
    final_classification["n_common_disciplines"] >= 2
)

project_metadata = project_level.drop_duplicates("project_id").copy()
project_metadata = project_metadata.drop(
    columns=[
        c
        for c in (
            discipline_classes
            + ["n_common_disciplines", "project_interdisciplinary", "label_source"]
        )
        if c in project_metadata.columns
    ],
    errors="ignore",
)

final_projects_1854 = project_metadata.merge(
    final_classification,
    on="project_id",
    how="left",
    validate="one_to_one",
)

assert len(final_projects_1854) == final_projects_1854["project_id"].nunique()
assert not final_projects_1854["label_source"].isna().any()

final_projects_1854.to_csv(
    RESULTS_DIR / "final_ce_projects_1854_disciplines.csv",
    index=False,
)

print("Final projects:", len(final_projects_1854))
print(final_projects_1854["label_source"].value_counts().to_string())

## 4. RQ1 — Funding distribution

Funding analysis is restricted to projects with usable positive observed funding values. For multidisciplinary projects, funding is allocated fractionally across assigned disciplines to avoid double counting.

In [ ]:
projects = final_projects_1854.copy()
projects["value_pounds"] = pd.to_numeric(projects["value_pounds"], errors="coerce")

if "funding_data_available" in projects.columns:
    usable_funding = (
        projects["funding_data_available"].fillna(False).astype(bool)
        & (projects["value_pounds"] > 0)
    )
else:
    usable_funding = projects["value_pounds"] > 0

funded = projects.loc[usable_funding].copy()

funding_summary = pd.Series({
    "projects": len(funded),
    "total_funding": funded["value_pounds"].sum(),
    "mean_funding": funded["value_pounds"].mean(),
    "median_funding": funded["value_pounds"].median(),
    "skewness": funded["value_pounds"].skew(),
})

funding_summary

In [ ]:
# Fractional disciplinary funding
funding_rows = []

for _, row in funded.iterrows():
    active = [d for d in discipline_classes if int(row[d]) == 1]
    if not active:
        continue

    share = row["value_pounds"] / len(active)
    for discipline in active:
        funding_rows.append({
            "project_id": row["project_id"],
            "discipline": discipline,
            "fractional_funding": share,
            "full_count_funding": row["value_pounds"],
        })

funding_long = pd.DataFrame(funding_rows)

fractional_summary = (
    funding_long.groupby("discipline")
    .agg(
        n_projects=("project_id", "nunique"),
        fractional_funding=("fractional_funding", "sum"),
    )
    .reset_index()
)

fractional_summary["fractional_funding_million"] = (
    fractional_summary["fractional_funding"] / 1_000_000
)
fractional_summary["funding_share_%"] = (
    fractional_summary["fractional_funding"]
    / fractional_summary["fractional_funding"].sum()
    * 100
)

median_by_discipline = {}
for discipline in discipline_classes:
    vals = funded.loc[funded[discipline].astype(int) == 1, "value_pounds"]
    median_by_discipline[discipline] = vals.median()

fractional_summary["median_project_funding"] = (
    fractional_summary["discipline"].map(median_by_discipline)
)

fractional_summary = fractional_summary.sort_values(
    "fractional_funding", ascending=False
).reset_index(drop=True)

fractional_summary.to_csv(
    RESULTS_DIR / "rq1_funding_by_discipline.csv",
    index=False,
)

fractional_summary

In [ ]:
# Funding by lead organisation
org = funded.copy()
org["lead_organisation_clean"] = (
    org["lead_organisation"]
    .fillna("Unknown")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

organisation_summary = (
    org.groupby("lead_organisation_clean")
    .agg(
        n_projects=("project_id", "nunique"),
        total_funding=("value_pounds", "sum"),
        median_funding=("value_pounds", "median"),
    )
    .reset_index()
)

organisation_summary["funding_share_%"] = (
    organisation_summary["total_funding"] / funded["value_pounds"].sum() * 100
)
organisation_summary["total_funding_million"] = (
    organisation_summary["total_funding"] / 1_000_000
)

organisation_summary = organisation_summary.sort_values(
    "total_funding", ascending=False
).reset_index(drop=True)
organisation_summary["cumulative_share_%"] = organisation_summary["funding_share_%"].cumsum()

organisation_summary.to_csv(
    RESULTS_DIR / "rq1_funding_by_lead_organisation.csv",
    index=False,
)

organisation_summary.head(20)

## 5. H1 — Disciplinary differences in project funding

Two OLS models use log-transformed observed project funding. HC3 heteroskedasticity-robust standard errors are reported. The adjusted model controls for lead funder and project start year; funders with fewer than 10 projects in the regression sample are collapsed into `Other funders`.

In [ ]:
h1 = funded.copy()

if "start_year" not in h1.columns:
    if "fund_start" not in h1.columns:
        raise KeyError("Need either start_year or fund_start for the adjusted H1 model.")
    h1["start_year"] = pd.to_datetime(h1["fund_start"], errors="coerce").dt.year

h1["log_funding"] = np.log(h1["value_pounds"])

discipline_short = {
    "Engineering & Manufacturing": "disc_engineering",
    "Materials & Chemistry": "disc_materials",
    "Environmental & Earth Sciences": "disc_environmental",
    "Life, Agricultural & Health Sciences": "disc_life",
    "Social Sciences, Business & Economics": "disc_social",
    "Energy": "disc_energy",
    "Computer & Quantitative Sciences": "disc_computer",
    "Arts, Humanities & Behavioural Sciences": "disc_arts",
}

for original, short in discipline_short.items():
    h1[short] = h1[original].astype(int)

disc_vars = list(discipline_short.values())

model1 = smf.ols(
    "log_funding ~ " + " + ".join(disc_vars),
    data=h1,
).fit(cov_type="HC3")

h1_model2 = h1[h1["start_year"].notna() & h1["lead_funder"].notna()].copy()

funder_counts = h1_model2["lead_funder"].value_counts()
major_funders = funder_counts[funder_counts >= 10].index
h1_model2["funder_group"] = np.where(
    h1_model2["lead_funder"].isin(major_funders),
    h1_model2["lead_funder"],
    "Other funders",
)

model2 = smf.ols(
    "log_funding ~ "
    + " + ".join(disc_vars)
    + " + C(funder_group) + start_year",
    data=h1_model2,
).fit(cov_type="HC3")

rows = []
for original, var in discipline_short.items():
    beta1 = float(model1.params[var])
    beta2 = float(model2.params[var])
    rows.append({
        "discipline": original,
        "model1_beta": beta1,
        "model1_SE": float(model1.bse[var]),
        "model1_p": float(model1.pvalues[var]),
        "model1_pct_difference": (np.exp(beta1) - 1) * 100,
        "model2_beta": beta2,
        "model2_SE": float(model2.bse[var]),
        "model2_p": float(model2.pvalues[var]),
        "model2_pct_difference": (np.exp(beta2) - 1) * 100,
    })

h1_results = pd.DataFrame(rows)

restriction = " = 0, ".join(disc_vars) + " = 0"
wald1 = model1.wald_test(restriction, scalar=True)
wald2 = model2.wald_test(restriction, scalar=True)

print("Model 1: N =", int(model1.nobs), "R² =", round(model1.rsquared, 3),
      "W =", round(float(wald1.statistic), 3), "p =", float(wald1.pvalue))
print("Model 2: N =", int(model2.nobs), "R² =", round(model2.rsquared, 3),
      "W =", round(float(wald2.statistic), 3), "p =", float(wald2.pvalue))

h1_results.to_csv(
    RESULTS_DIR / "rq1_h1_discipline_funding_regression_final.csv",
    index=False,
)

h1_results.round(4)

## 6. RQ2 — Map publication outputs to the common disciplinary framework

In [ ]:
rq2_links = final_integrated.copy()

field_crosswalk = (
    openalex_field_table[["openalex_field", "common_discipline"]]
    .drop_duplicates()
    .copy()
)

rq2_links = rq2_links.merge(
    field_crosswalk,
    left_on="field",
    right_on="openalex_field",
    how="left",
).rename(columns={"common_discipline": "output_discipline"})

rq2_analysis_links = rq2_links[
    rq2_links["output_discipline"].notna()
].copy()

publication_disciplines = (
    rq2_analysis_links[["publication_id", "output_discipline"]]
    .drop_duplicates()
)

output_distribution = (
    publication_disciplines["output_discipline"]
    .value_counts()
    .rename_axis("output_discipline")
    .reset_index(name="n_publications")
)
output_distribution["publication_%"] = (
    output_distribution["n_publications"]
    / output_distribution["n_publications"].sum()
    * 100
)

print("Classified project-publication links:", len(rq2_analysis_links))
print("Projects:", rq2_analysis_links["project_id"].nunique())
print("Unique classified publications:", rq2_analysis_links["publication_id"].nunique())

rq2_analysis_links.to_csv(
    RESULTS_DIR / "rq2_project_output_discipline_links.csv",
    index=False,
)
output_distribution.to_csv(
    RESULTS_DIR / "rq2_output_discipline_distribution.csv",
    index=False,
)

output_distribution

## 7. Project-level disciplinary profiles and alignment

In [ ]:
disciplines = discipline_classes

links = (
    rq2_analysis_links[["project_id", "publication_id", "output_discipline"]]
    .drop_duplicates()
)

output_counts = pd.crosstab(links["project_id"], links["output_discipline"])
output_counts = output_counts.reindex(columns=disciplines, fill_value=0)

n_outputs = output_counts.sum(axis=1)
output_shares = output_counts.div(n_outputs, axis=0)

project_binary = (
    final_projects_1854.set_index("project_id")[disciplines]
    .astype(int)
)

project_binary = project_binary.loc[
    project_binary.index.intersection(output_shares.index)
]

# Equal fractional weight across each project's assigned disciplines.
project_weights = project_binary.div(
    project_binary.sum(axis=1),
    axis=0,
)

common_ids = project_weights.index.intersection(output_shares.index)
project_binary = project_binary.loc[common_ids]
project_weights = project_weights.loc[common_ids]
output_counts = output_counts.loc[common_ids]
output_shares = output_shares.loc[common_ids]

P = project_weights.to_numpy(dtype=float)
B = project_binary.to_numpy(dtype=float)
O = output_shares.to_numpy(dtype=float)

alignment_share = (B * O).sum(axis=1)

numerator = (P * O).sum(axis=1)
denominator = np.sqrt((P ** 2).sum(axis=1)) * np.sqrt((O ** 2).sum(axis=1))
cosine_similarity = np.divide(
    numerator,
    denominator,
    out=np.zeros_like(numerator, dtype=float),
    where=denominator != 0,
)

project_sets = B > 0
output_sets = O > 0
intersection = np.logical_and(project_sets, output_sets).sum(axis=1)
union = np.logical_or(project_sets, output_sets).sum(axis=1)
jaccard_similarity = np.divide(
    intersection,
    union,
    out=np.zeros_like(intersection, dtype=float),
    where=union != 0,
)

rq2_project_profiles = pd.DataFrame({
    "project_id": common_ids,
    "n_classified_outputs": output_counts.sum(axis=1).to_numpy(),
    "alignment_share": alignment_share,
    "cosine_similarity": cosine_similarity,
    "jaccard_similarity": jaccard_similarity,
    "n_project_disciplines": project_binary.sum(axis=1).to_numpy(),
    "n_output_disciplines": (output_counts > 0).sum(axis=1).to_numpy(),
    "n_output_disciplines_10pct": (output_shares >= 0.10).sum(axis=1).to_numpy(),
})

rq2_project_profiles.to_csv(
    RESULTS_DIR / "rq2_project_level_disciplinary_profiles.csv",
    index=False,
)

rq2_project_profiles[
    ["alignment_share", "cosine_similarity", "jaccard_similarity"]
].describe()

## 8. Transition matrix and H2 permutation test

Each project contributes equal total weight to the transition matrix. For H2, complete output disciplinary profiles are randomly reassigned across projects 10,000 times.

In [ ]:
transition = np.zeros((len(disciplines), len(disciplines)))

for i in range(len(common_ids)):
    transition += np.outer(P[i], O[i])

transition_df = pd.DataFrame(
    transition,
    index=disciplines,
    columns=disciplines,
)

transition_row_pct = (
    transition_df.div(transition_df.sum(axis=1), axis=0) * 100
)

def cosine_rows(A, B):
    num = (A * B).sum(axis=1)
    den = np.sqrt((A ** 2).sum(axis=1)) * np.sqrt((B ** 2).sum(axis=1))
    return np.divide(
        num,
        den,
        out=np.zeros_like(num, dtype=float),
        where=den != 0,
    )

observed_cosine = cosine_rows(P, O).mean()
observed_alignment = (B * O).sum(axis=1).mean()

N_PERM = 10_000
rng = np.random.default_rng(42)
perm_cosine = np.empty(N_PERM)
perm_alignment = np.empty(N_PERM)

for i in range(N_PERM):
    idx = rng.permutation(len(common_ids))
    O_perm = O[idx]
    perm_cosine[i] = cosine_rows(P, O_perm).mean()
    perm_alignment[i] = (B * O_perm).sum(axis=1).mean()

p_cosine = (np.sum(perm_cosine >= observed_cosine) + 1) / (N_PERM + 1)
p_alignment = (np.sum(perm_alignment >= observed_alignment) + 1) / (N_PERM + 1)

permutation_summary = pd.DataFrame([
    {
        "metric": "cosine_similarity",
        "n_projects": len(common_ids),
        "observed": observed_cosine,
        "random_expectation": perm_cosine.mean(),
        "difference_above_chance": observed_cosine - perm_cosine.mean(),
        "empirical_p": p_cosine,
    },
    {
        "metric": "alignment_share",
        "n_projects": len(common_ids),
        "observed": observed_alignment,
        "random_expectation": perm_alignment.mean(),
        "difference_above_chance": observed_alignment - perm_alignment.mean(),
        "empirical_p": p_alignment,
    },
])

transition_row_pct.to_csv(
    RESULTS_DIR / "rq2_transition_matrix_row_percent.csv"
)
permutation_summary.to_csv(
    RESULTS_DIR / "rq2_h2_permutation_test.csv",
    index=False,
)

permutation_summary

## 9. Interdisciplinarity analysis

In [ ]:
inter = rq2_project_profiles.copy()

inter["project_interdisciplinary"] = inter["n_project_disciplines"] >= 2
inter["output_interdisciplinary_10pct"] = inter["n_output_disciplines_10pct"] >= 2
inter["output_interdisciplinary_any"] = inter["n_output_disciplines"] >= 2

transition_counts = pd.crosstab(
    inter["project_interdisciplinary"],
    inter["output_interdisciplinary_10pct"],
).reindex(index=[False, True], columns=[False, True], fill_value=0)

# McNemar exact test
mcnemar_result = mcnemar(transition_counts.to_numpy(), exact=True)

# Breadth comparison
wilcoxon_result = wilcoxon(
    inter["n_project_disciplines"],
    inter["n_output_disciplines_10pct"],
    zero_method="wilcox",
)

inter["breadth_change"] = (
    inter["n_output_disciplines_10pct"] - inter["n_project_disciplines"]
)

interdisciplinarity_summary = pd.Series({
    "project_interdisciplinary_n": int(inter["project_interdisciplinary"].sum()),
    "output_interdisciplinary_n": int(inter["output_interdisciplinary_10pct"].sum()),
    "mcnemar_p": float(mcnemar_result.pvalue),
    "mean_project_breadth": inter["n_project_disciplines"].mean(),
    "mean_output_breadth": inter["n_output_disciplines_10pct"].mean(),
    "wilcoxon_p": float(wilcoxon_result.pvalue),
    "n_narrower": int((inter["breadth_change"] < 0).sum()),
    "n_same": int((inter["breadth_change"] == 0).sum()),
    "n_broader": int((inter["breadth_change"] > 0).sum()),
})

transition_counts.to_csv(
    RESULTS_DIR / "rq2_interdisciplinarity_transition_summary.csv"
)
interdisciplinarity_summary

## 10. Robustness to project maturity

In [ ]:
project_meta = (
    final_projects_1854[
        ["project_id", "fund_start", "fund_end", "status", "lead_funder"]
    ]
    .drop_duplicates("project_id")
    .copy()
)

project_meta["fund_start"] = pd.to_datetime(project_meta["fund_start"], errors="coerce")
project_meta["start_year"] = project_meta["fund_start"].dt.year

rq2_robust = rq2_project_profiles[["project_id"]].merge(
    project_meta,
    on="project_id",
    how="left",
)

def run_alignment_permutation(project_ids, label, n_perm=10_000, seed=42):
    ids = (
        project_weights.index
        .intersection(pd.Index(project_ids))
        .intersection(output_shares.index)
    )

    P_sub = project_weights.loc[ids, disciplines].to_numpy(dtype=float)
    B_sub = project_binary.loc[ids, disciplines].to_numpy(dtype=float)
    O_sub = output_shares.loc[ids, disciplines].to_numpy(dtype=float)

    obs_cos = cosine_rows(P_sub, O_sub).mean()
    obs_align = (B_sub * O_sub).sum(axis=1).mean()

    rng = np.random.default_rng(seed)
    perm_cos = np.empty(n_perm)
    perm_align = np.empty(n_perm)

    for i in range(n_perm):
        idx = rng.permutation(len(ids))
        O_perm = O_sub[idx]
        perm_cos[i] = cosine_rows(P_sub, O_perm).mean()
        perm_align[i] = (B_sub * O_perm).sum(axis=1).mean()

    return {
        "sample": label,
        "n_projects": len(ids),
        "observed_cosine": obs_cos,
        "random_cosine": perm_cos.mean(),
        "cosine_difference": obs_cos - perm_cos.mean(),
        "cosine_p": (np.sum(perm_cos >= obs_cos) + 1) / (n_perm + 1),
        "observed_alignment": obs_align,
        "random_alignment": perm_align.mean(),
        "alignment_difference": obs_align - perm_align.mean(),
        "alignment_p": (np.sum(perm_align >= obs_align) + 1) / (n_perm + 1),
    }

samples = [
    (rq2_robust["project_id"], "Full RQ2 sample"),
    (
        rq2_robust.loc[rq2_robust["start_year"] <= 2022, "project_id"],
        "Started by 2022",
    ),
    (
        rq2_robust.loc[
            rq2_robust["status"].astype(str).str.lower().eq("closed"),
            "project_id",
        ],
        "Closed projects",
    ),
    (
        rq2_robust.loc[
            (rq2_robust["start_year"] <= 2022)
            & rq2_robust["status"].astype(str).str.lower().eq("closed"),
            "project_id",
        ],
        "Closed and started by 2022",
    ),
]

robustness_results = pd.DataFrame([
    run_alignment_permutation(ids, label)
    for ids, label in samples
])

robustness_results.to_csv(
    RESULTS_DIR / "rq2_alignment_maturity_robustness.csv",
    index=False,
)

robustness_results.round(4)

## 11. Final figures

The cells below regenerate the six main dissertation figures from the cleaned analytical objects.

In [ ]:
# Figure 1 — projects and observed funding by start year
fig1 = final_projects_1854.copy()
fig1["fund_start"] = pd.to_datetime(fig1["fund_start"], errors="coerce")
fig1["start_year"] = fig1["fund_start"].dt.year
fig1["value_pounds"] = pd.to_numeric(fig1["value_pounds"], errors="coerce")

plot_years = fig1[fig1["start_year"].notna() & (fig1["start_year"] <= 2025)].copy()

projects_by_year = plot_years.groupby("start_year")["project_id"].nunique()
funding_by_year = (
    plot_years.loc[
        (plot_years["value_pounds"] > 0)
        & (
            plot_years["funding_data_available"].fillna(False).astype(bool)
            if "funding_data_available" in plot_years.columns
            else True
        )
    ]
    .groupby("start_year")["value_pounds"]
    .sum()
    / 1_000_000
)

years = sorted(set(projects_by_year.index).union(funding_by_year.index))

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(years, [projects_by_year.get(y, 0) for y in years])
ax.set_xlabel("Project start year")
ax.set_ylabel("Number of projects")
ax.set_title("A. Circular economy projects by start year")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_1A_projects_by_start_year.svg", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(years, [funding_by_year.get(y, 0) for y in years])
ax.set_xlabel("Project start year")
ax.set_ylabel("Observed funding (£ million)")
ax.set_title("B. Observed project funding by start year")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_1B_funding_by_start_year.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2 — fractional funding by discipline
plot_df = fractional_summary.sort_values(
    "fractional_funding_million",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(plot_df["discipline"], plot_df["fractional_funding_million"])

for bar, value, pct in zip(
    bars,
    plot_df["fractional_funding_million"],
    plot_df["funding_share_%"],
):
    ax.text(
        bar.get_width() + 4,
        bar.get_y() + bar.get_height() / 2,
        f"£{value:.1f}m ({pct:.1f}%)",
        va="center",
        fontsize=9,
    )

ax.set_xlabel("Fractionally allocated observed funding (£ million)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_2_fractional_funding_by_discipline.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3 — top 15 named lead organisations by funding
top_orgs = organisation_summary[
    organisation_summary["lead_organisation_clean"] != "Unknown"
].head(15).sort_values("total_funding_million", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top_orgs["lead_organisation_clean"], top_orgs["total_funding_million"])

for bar, value, pct in zip(
    bars,
    top_orgs["total_funding_million"],
    top_orgs["funding_share_%"],
):
    ax.text(
        bar.get_width() + 0.8,
        bar.get_y() + bar.get_height() / 2,
        f"£{value:.1f}m ({pct:.1f}%)",
        va="center",
        fontsize=8,
    )

ax.set_xlabel("Observed funding (£ million)")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_3_top_lead_organisations.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 4 — research outputs by discipline
fig4_df = output_distribution.copy()
fig4_df = fig4_df.sort_values("n_publications", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(fig4_df["output_discipline"], fig4_df["n_publications"])

for bar, count, pct in zip(
    bars,
    fig4_df["n_publications"],
    fig4_df["publication_%"],
):
    ax.text(
        bar.get_width() + 25,
        bar.get_y() + bar.get_height() / 2,
        f"{int(count):,} ({pct:.1f}%)",
        va="center",
        fontsize=9,
    )

ax.set_xlabel("Number of classified publications")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_4_research_outputs_by_discipline.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 5 — project-to-output disciplinary transition heatmap
matrix = transition_row_pct.to_numpy()

fig, ax = plt.subplots(figsize=(10, 7))
image = ax.imshow(matrix, aspect="auto")
fig.colorbar(image, ax=ax, label="Share of disciplinary output flow (%)")

ax.set_xticks(range(len(disciplines)))
ax.set_xticklabels(disciplines, rotation=55, ha="right", fontsize=8)
ax.set_yticks(range(len(disciplines)))
ax.set_yticklabels(disciplines, fontsize=8)
ax.set_xlabel("Output discipline")
ax.set_ylabel("Project discipline")

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_5_project_output_transition_heatmap.svg", bbox_inches="tight")
plt.show()

In [ ]:
# Figure 6 — interdisciplinarity transitions
table = transition_counts.copy()

disc_total = table.loc[False].sum()
inter_total = table.loc[True].sum()

disc_values = [
    table.loc[False, False] / disc_total * 100,
    table.loc[False, True] / disc_total * 100,
]
inter_values = [
    table.loc[True, False] / inter_total * 100,
    table.loc[True, True] / inter_total * 100,
]

x = np.arange(2)
bottom = np.array([0.0, 0.0])

fig, ax = plt.subplots(figsize=(7, 5))

disciplinary_output = np.array([disc_values[0], inter_values[0]])
interdisciplinary_output = np.array([disc_values[1], inter_values[1]])

ax.bar(x, disciplinary_output)
ax.bar(x, interdisciplinary_output, bottom=disciplinary_output)

ax.set_xticks(x)
ax.set_xticklabels(["Disciplinary projects", "Interdisciplinary projects"])
ax.set_ylabel("Share of projects (%)")
ax.set_xlabel("Project-stage disciplinary profile")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "Figure_6_interdisciplinarity_transitions.svg", bbox_inches="tight")
plt.show()

## 12. Key result files generated

After a successful run, the main files in `results/` include:

- `final_ce_projects_1854_disciplines.csv`
- `rq1_funding_by_discipline.csv`
- `rq1_funding_by_lead_organisation.csv`
- `rq1_h1_discipline_funding_regression_final.csv`
- `rq2_project_output_discipline_links.csv`
- `rq2_project_level_disciplinary_profiles.csv`
- `rq2_transition_matrix_row_percent.csv`
- `rq2_h2_permutation_test.csv`
- `rq2_interdisciplinarity_transition_summary.csv`
- `rq2_alignment_maturity_robustness.csv`

This repository notebook is intended as a readable and reproducible companion to the dissertation rather than a complete archive of every exploratory step performed during the project.